In [1]:
from time import time
from typing import List
from pathlib import Path

from ctranslate2 import Translator
from datasets import load_dataset
from pydantic import DirectoryPath
from sacrebleu import corpus_bleu, corpus_chrf
from transformers import AutoTokenizer

/home/mark/miniconda3/envs/qmt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class OpusmtTranslator:
    def __init__(self, model_path: DirectoryPath, model_id: str, ct2_device: str = "auto", compute_type: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.translator = Translator(
            model_path, device=ct2_device, compute_type=compute_type)

    def tokenize(self, x: List[str]):
        return self.tokenizer.convert_ids_to_tokens(self.tokenizer.encode(x))

    def translate(self, x, beam_size: int = 5):
        if isinstance(x, str):
            return_string = True
            x = [x]
        else:
            return_string = False

        x = [self.tokenize(i) for i in x]
        ret = self.translator.translate_batch(
            x, max_batch_size=32, beam_size=beam_size)
        ret = [self.tokenizer.convert_tokens_to_string(
            i.hypotheses[0]) for i in ret]
        if return_string:
            return ret[0]
        else:
            return ret

In [3]:
class NLLBTranslator:
    def __init__(self, model_path: DirectoryPath, model_id: str, src_lang: str, tgt_lang: str, ct2_device: str = "auto", compute_type: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, src_lang=src_lang)
        self.translator = Translator(
            model_path, device=ct2_device, compute_type=compute_type)
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang

    def tokenize(self, x: List[str]):
        return self.tokenizer.convert_ids_to_tokens(self.tokenizer.encode(x))

    def translate(self, x: str | List[str], beam_size: int = 5):
        if isinstance(x, str):
            return_string = True
            x = [x]
        else:
            return_string = False

        x = [self.tokenize(i) for i in x]
        ret = self.translator.translate_batch(
            x, target_prefix=[[self.tgt_lang]]*len(x), max_batch_size=32, beam_size=5
        )
        ret = [self.tokenizer.convert_tokens_to_string(
            i.hypotheses[0][1:]) for i in ret]
        if return_string:
            return ret[0]
        else:
            return ret

## Flores Data

In [4]:
src_lang_flores = "isl_Latn"
tgt_lang_flores = "eng_Latn"

src_lang = "Icelandic"
tgt_lang = "English"

ct2_device = "cuda"
ct2_compute_type = "auto"

tgt = load_dataset("openlanguagedata/flores_plus",
                   tgt_lang_flores, split="devtest")
src = load_dataset("openlanguagedata/flores_plus",
                   src_lang_flores, split="devtest")

src = [i['text'] for i in src]
tgt = [i['text'] for i in tgt]

## Opus-MT

To download and convert and OpusMT model:

```bash
ct2-transformers-converter --model Helsinki-NLP/opus-mt-tc-big-zle-en --output_dir ./ct2-models/opus-mt-tc-big-zle-en-ct2 --force

ct2-transformers-converter --model Helsinki-NLP/opus-mt-en-uk --output_dir ./ct2-models/opus-mt-en-uk-ct2 --force
ct2-transformers-converter --model Helsinki-NLP/opus-mt-is-en --output_dir ./ct2-models/opus-mt-is-en-ct2 --force

```

In [9]:
# opusmt_model = "../ct2-models/opus-mt-uk-en-ct2/"
# opusmt_id = "Helsinki-NLP/opus-mt-uk-en"
# opusmt_model = "../ct2-models/opus-mt-zle-en-big-ct2"
# opusmt_id = "Helsinki-NLP/opus-mt-tc-big-zle-en"

#opusmt_model = "~/mt/ct2-models/opus-mt-en-uk-ct2/"
#opusmt_id = "Helsinki-NLP/opus-mt-en-uk"

opusmt_model = "~/code/ct2-models/opus-mt-is-en-ct2/"
opusmt_id = "Helsinki-NLP/opus-mt-is-en"

# opusmt_model = "~/mt/ct2-models/opus-mt-en-uk-ct2"
# opusmt_id = "Helsinki-NLP/opus-mt-en-uk"


t = OpusmtTranslator(str(Path(opusmt_model).expanduser()), opusmt_id,
                     ct2_device=ct2_device, compute_type=ct2_compute_type)

In [10]:
t1 = time()
if opusmt_id == "Helsinki-NLP/opus-mt-tc-big-en-zle":
    mt = t.translate([">>ukr<< "+i for i in src], beam_size=5)
else:
    mt = t.translate(src, beam_size=5)
t2 = time()
del t
opus_time = t2 - t1

In [11]:
opus_bleu = corpus_bleu(mt, [tgt]).score
opus_bleu

25.262893626149957

In [12]:
opus_chrf = corpus_chrf(mt, [tgt]).score
opus_chrf

51.44330889840445

## NLLB

In [43]:
t = NLLBTranslator(str(Path("~/code/ct2-models/nllb-200-distilled-1.3B-ct2/").expanduser()),
                   "facebook/nllb-200-distilled-1.3B", src_lang_flores, tgt_lang_flores, ct2_device, compute_type=ct2_compute_type)

In [44]:
t1 = time()
mt = t.translate(src, beam_size=5)
t2 = time()
del t
nllb_time = t2 - t1
nllb_time

TypeError: 'module' object is not callable. Did you mean: 'time.time(...)'?

In [ ]:
nllb_bleu = corpus_bleu(mt, [tgt]).score
nllb_bleu

In [ ]:
nllb_chrf = corpus_chrf(mt, [tgt]).score
nllb_chrf

## QuickMT

In [ ]:
from quickmt import Translator as QuickmtTranslator
ct2_compute_type="float16"

In [ ]:
# t = QuickmtTranslator("~/Downloads/isen-vastai",
#                device=ct2_device, compute_type=ct2_compute_type)

In [ ]:
t = QuickmtTranslator(str(Path("~/Downloads/isen-vastai").expanduser()),
               device=ct2_device, compute_type=ct2_compute_type)

In [ ]:
src[1]

In [ ]:
t(src[1], beam_size=5)

In [ ]:
t(src[1], sampling_temperature=1.2, beam_size=1, sampling_topk=50, sampling_topp=0.9)

In [ ]:
t1 = time()
mt = t.translate(src, beam_size=5, max_batch_size=32)
t2 = time()
del t
quickmt_time = t2 - t1
quickmt_time

In [ ]:
quickmt_bleu = corpus_bleu(mt, [tgt]).score
quickmt_bleu

In [ ]:
quickmt_chrf = corpus_chrf(mt, [tgt]).score
quickmt_chrf

## Results

Print out markdown table

In [ ]:
print(f"| model | time | bleu | chrf |")
print(f"| ----------- | ------- | ------ | -------|")
print(f"| quickmt-en-uk | {(quickmt_time):.2f} | {
      quickmt_bleu:.2f}| {quickmt_chrf:.2f} |")
print(f"| {opusmt_id} | {(opus_time):.2f} | {
      opus_bleu:.2f}| {opus_chrf:.2f} |")
print(f"| facebook/nllb-200-distilled-1.3B | {
      (nllb_time):.2f} | {nllb_bleu:.2f}| {nllb_chrf:.2f} |")

| model | time | bleu | chrf |
| ----------- | ------- | ------ | -------|
| quickmt-is-en | 1.24 | 35.91| 60.83 |
| Helsinki-NLP/opus-mt-is-en | 2.33 | 25.26| 51.44 |
| facebook/nllb-200-distilled-1.3B | 18.17 | 32.79| 56.81 |
| CohereLabs/tiny-aya-global | 27.03 | 16.03| 40.63 |
| google/gemma-4-E2B-it | 47.66 | 28.55| 54.30 |
| Qwen/Qwen3.5-4B | 65.94 | 27.53| 53.84 |


| model | time | bleu | chrf |
| ----------- | ------- | ------ | -------|
| quickmt_uk_en | 1.12 | 39.88| 65.65 |
| Helsinki-NLP/opus-mt-tc-big-zle-en | 3.03 | 39.07| 64.98 |
| facebook/nllb-200-distilled-1.3B | 17.25 | 33.59| 63.50 |
| CohereLabs/tiny-aya-global | 23.36 | 37.60| 63.73 |
| google/gemma-4-E2B-it | 47.71 | 37.22| 63.75 |

| model                              | time  | bleu  | chrf  |
|------------------------------------|-------|-------|-------|
| quickmt-en-uk                      | 1.20  | 32.44 | 60.52 |
| Helsinki-NLP/opus-mt-tc-big-en-zle | 3.78  | 31.98 | 60.08 |
| facebook/nllb-200-distilled-1.3B   | 19.23 | 26.31 | 56.91 |
| CohereLabs/tiny-aya-global         | 35.37 | 27.45 | 56.22 |
| google/gemma-4-E2B-it              | 70.78 | 29.02 | 57.54 |


## Benchmark LLMs running With vLLM

```bash
# Create env
conda create -n vllm python=3.12 pip
conda activate vllm
pip install vllm openai tqdm huggingface_hub

# May be required to run latest models:
# pip install --upgrade transformers

# Log in, in case model is gated
hf auth login

# Launch LLM
vllm serve CohereLabs/tiny-aya-global --gpu_memory_utilization=0.85 --max_num_seqs=32 --max_model_len 4096
vllm serve google/gemma-4-E2B-it --gpu_memory_utilization=0.90 --max_num_seqs=8 --max_model_len 1024

# vllm serve google/gemma-4-E4B-it --gpu_memory_utilization=0.90 --max_num_seqs=1 --max_model_len 1024 --quantization fp8
# vllm serve CohereLabs/aya-23-8B --gpu_memory_utilization=0.90 --max_num_seqs=8 --max_model_len 1024 --quantization fp8

# These looked promising but don't seem to work very well
# vllm serve ministral/Ministral-3b-instruct --gpu_memory_utilization=0.90 --max_num_seqs=8 --max_model_len 1024
# vllm serve Qwen/Qwen3.5-4B --gpu_memory_utilization=0.90 --max_num_seqs=8 --max_model_len 1024 --quantization fp8 --reasoning-parser qwen3 --language-model-only

```

In [16]:
import asyncio
import os
import time
from concurrent.futures import ThreadPoolExecutor

import openai
from openai import OpenAI
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio

client = OpenAI(api_key="sk_foo", base_url="http://localhost:8000/v1", timeout=120)

model_id = client.models.list().data[0].id

In [17]:
def translate_gemma(x, src_lang: str = "uk", tgt_lang: str = "en"):
    """Sends a single async request to the OpenAI API."""
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                  "role": "user",
                  "content": f"<<<source>>>{src_lang}<<<target>>>{tgt_lang}<<<text>>>{x}"
                }
            ],
            temperature=0.0,
            max_tokens=512,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return ""
        
def translate(x, src_lang=src_lang, tgt_lang=tgt_lang):
    """Sends a single async request to the OpenAI API."""
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                    "role": "user",
                    "content": f"Translate the following into {tgt_lang}, without commentary or explanation.\n\n{x}",
                    #"content": f"You are a professional {src_lang}-to-{tgt_lang} translator. Your goal is to accurately convey the meaning and nuances of the original {src_lang} text while adhering to {tgt_lang} grammar, vocabulary, and cultural sensitivities. Produce only the {tgt_lang} translation, without any additional explanations or commentary. Retain the paragraph breaks (double new lines) from the input text. Please translate the following {src_lang} text into {tgt_lang}:\n\n{x}"
                }
            ],
            extra_body={
                #"top_k": 20,
                "chat_template_kwargs": {"enable_thinking": False},
            }, 
            temperature=0.0,
            max_tokens=512,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return ""

if model_id == "Infomaniak-AI/vllm-translategemma-4b-it":
    translate = translate_gemma


In [18]:
translate("Test")

'Test'

In [19]:
def translate_parallel(items):
    with ThreadPoolExecutor(max_workers=64) as executor:
        results = list(tqdm(executor.map(translate, items), total=len(items)))
    return results

t1 = time.time()
mt = translate_parallel(src)
t2 = time.time()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1012/1012 [01:05<00:00, 15.45it/s]


In [20]:
mt[:10]

['"We now have four-month-old mice that are not diabetic, but were diabetic before," he added.',
 'Dr. Ehud Ur, a professor of medicine at Dalhousie University in Halifax, Nova Scotia, and chair of the clinical science department of the Canadian Diabetes Association, noted that the research was only recently published.',
 'One and other experts doubt whether diabetes can be cured and point out that these findings have no significance for those who already have type 1 diabetes.',
 'Sara Danius, the secretary of the Nobel Committee for Literature at the Swedish Academy, publicly announced on a Swedish Radio program that the committee could not contact Bob Dylan directly regarding the 2016 Nobel Prize in Literature, as they had given up on reaching him.',
 'Danius said: "We are not doing anything right now. I have called and sent emails to his closest collaborator and received very friendly responses. At this moment, that is quite enough."',
 'Before CEO Jamie Siminoff of Ring had said th

In [21]:
llm_time = t2 - t1
llm_bleu = corpus_bleu(mt, [tgt]).score
llm_chrf = corpus_chrf(mt, [tgt]).score

In [22]:
print(f"| model | time | bleu | chrf |")
print(f"| ----------- | ------- | ------ | -------|")
print(f"| {model_id} | {(llm_time):.2f} | {
      llm_bleu:.2f}| {llm_chrf:.2f} |")

| model | time | bleu | chrf |
| ----------- | ------- | ------ | -------|
| Qwen/Qwen3.5-4B | 65.94 | 27.53| 53.84 |
